# 12 - Optimization solver (Stage 14)

## 1. Objective
Use integer linear programming (PuLP + CBC) to choose which outages to repair in one day under a crew budget and borough minimum quotas. All logic is in `src/optimization/ilp_solver.py`; this notebook calls it.

**What 'benefit' means.** The Guide's `sum(x_i * tau_net_i)` cannot be applied literally: `tau_net` is one global Stage 11 estimate. The outage-specific quantity the project derived from it is the Stage 12 `priority_score`, so `benefit_i = priority_score_i` (index points). The optimum is the maximum total index of the repaired outages. It is not a count of crimes prevented and not an estimate of outage-level causal effects.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.optimization import ilp_solver as ilp
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

## 2-3. Stage 13 scored outages and schema

In [2]:
frame = ilp.load_scored_outages()
print(frame.shape)
print(frame[list(ilp.REQUIRED_COLUMNS)].dtypes)
cands = ilp.candidates(frame)
print('candidates (scored):', len(cands))
print(cands.borough.value_counts())

(107731, 55)
unique_key                 int64
borough                      str
created_date      datetime64[us]
priority_score           float64
scored                      bool
dtype: object


candidates (scored): 103830
borough
BROOKLYN         32423
QUEENS           30634
BRONX            18475
MANHATTAN        11624
STATEN ISLAND    10657
Unspecified         17
Name: count, dtype: int64


## 4-7. Variables, objective, budget and fairness constraints
- `x_i` in {0,1} for every candidate outage: 1 = repair, 0 = defer.
- Maximise `sum(benefit_i * x_i)`.
- Daily budget: `sum(repair_cost_i * x_i) <= daily_budget`. The repository defines only the Stage 13 capacity K = 20 repairs/day, so `repair_cost = 1` for every outage and `daily_budget = 20` (no cost field exists).
- Borough fairness: `sum(x_i for i in borough b) >= quota_b`. No quotas exist in the repository; the Stage 14 convention is `quota_b = floor(0.5 * 20 * share_b)`, where `share_b` is the borough's share of the backlog. Explicit overrides are possible. Feasibility is checked before solving.
All values are function/CLI parameters.

In [3]:
budget = ilp.DEFAULT_DAILY_BUDGET
quotas = ilp.resolve_quotas(cands, budget, ilp.DEFAULT_QUOTA_FRACTION)
print('daily budget:', budget, '| repair cost:', ilp.DEFAULT_REPAIR_COST)
print('quotas:', quotas, '| sum:', sum(quotas.values()))
ilp.check_feasibility(cands, quotas, budget); print('quotas feasible')

daily budget: 20.0 | repair cost: 1.0
quotas: {'BRONX': 1, 'BROOKLYN': 3, 'MANHATTAN': 1, 'QUEENS': 2, 'STATEN ISLAND': 1, 'Unspecified': 0} | sum: 8
quotas feasible


## 8-9. Build and solve (CBC)

In [4]:
plan, summary = ilp.optimise(frame)
print('solver status:', summary['solver_status'], '| pulp', summary['pulp_version'])
print('candidates', summary['n_candidates'], '| selected', summary['n_selected'], '| deferred', summary['n_deferred'])
print('objective', round(summary['objective_value'], 4))

solver status: Optimal | pulp 3.3.2
candidates 103830 | selected 20 | deferred 103810
objective 1152.8322


## 10. Optimal repair plan

In [5]:
plan[plan.selected_for_repair][['optimization_rank','outage_id','borough','created_date','priority_score','benefit','repair_cost','decision']]

,optimization_rank,outage_id,borough,created_date,priority_score,benefit,repair_cost,decision
0,1,57807769,MANHATTAN,2023-06-05 17:37:00,100.000000,100.000000,1.0,repair
1,2,65567892,QUEENS,2025-07-15 09:19:00,88.530035,88.530035,1.0,repair
2,3,57943263,MANHATTAN,2023-06-19 22:44:00,82.378708,82.378708,1.0,repair
3,4,56570130,MANHATTAN,2023-01-19 08:41:00,79.209077,79.209077,1.0,repair
4,5,62152298,BROOKLYN,2024-08-17 21:59:00,71.328830,71.328830,1.0,repair
5,6,62645624,MANHATTAN,2024-10-03 16:50:00,66.224219,66.224219,1.0,repair
6,7,65912548,MANHATTAN,2025-08-21 12:01:00,58.615669,58.615669,1.0,repair
7,8,64371693,MANHATTAN,2025-03-16 21:25:00,57.389564,57.389564,1.0,repair
8,9,54091601,MANHATTAN,2022-05-05 09:33:00,54.837946,54.837946,1.0,repair
9,10,61473105,QUEENS,2024-06-13 16:44:00,54.154837,54.154837,1.0,repair


## 11. Borough allocation and quota compliance

In [6]:
alloc = pd.DataFrame({'quota': pd.Series(summary['quotas']), 'selected': pd.Series(summary['borough_allocation']),
                      'unconstrained': pd.Series(summary['unconstrained_borough_allocation'])}).fillna(0).astype(int)
alloc['quota_met'] = alloc.selected >= alloc.quota
alloc

,quota,selected,unconstrained,quota_met
BRONX,1,1,0,True
BROOKLYN,3,3,1,True
MANHATTAN,1,13,15,True
QUEENS,2,2,4,True
STATEN ISLAND,1,1,0,True
Unspecified,0,0,0,True


## 12. Budget utilisation and the price of fairness

In [7]:
print(f"budget used {summary['budget_used']:g} of {summary['parameters']['daily_budget']:g}; remaining {summary['budget_remaining']:g}")
print(f"objective with quotas {summary['objective_value']:.4f} | without quotas {summary['unconstrained_objective']:.4f} | "
      f"price of fairness {summary['price_of_fairness']:.4f} ({100*summary['price_of_fairness']/summary['unconstrained_objective']:.1f}%)")

budget used 20 of 20; remaining 0
objective with quotas 1152.8322 | without quotas 1208.6336 | price of fairness 55.8014 (4.6%)


## 13. Selected vs deferred

In [8]:
plan.groupby('decision').priority_score.describe()[['count','mean','min','50%','max']]

,count,mean,min,50%,max
decision,,,,,
defer,103810.0,0.512418,0.00000,0.031804,48.847645
repair,20.0,57.641609,9.31896,53.443601,100.000000


In [9]:
saved = pd.read_csv(ilp.OUTPUT_DIR / ilp.PLAN_FILENAME)
print('matches saved plan:', saved.outage_id.tolist() == plan.outage_id.tolist() and (saved.selected_for_repair == plan.selected_for_repair).all())

matches saved plan: True


## 14. Limitations and assumptions
- The objective is the Stage 12 index (not a probability or crime count); tau_net is constant and not statistically distinguishable from zero, so the ILP effectively picks high recent-crime, long-duration outages, subject to the quotas.
- `repair_cost = 1` and `daily_budget = 20` are conventions (the Stage 13 capacity), not real costs; quotas are a Stage 14 convention, not an agency rule.
- One day's dispatch from the whole scored backlog; no arrivals, travel, crew skills or multi-day scheduling. Durations are retrospective (Stage 12).
- With uniform costs and these constraints the ILP is small; the result is checked against brute force on small cases in the validator.
- Ties between equal benefits are broken deterministically toward earlier `created_date`.